# Fusión de Datos v7 · Calidad y Preprocesamiento
**Equipo:** Calidad de Datos (Carlo Kiliano Ferrera, José Julian Pérez, Aldo Sebastián Altamirano)
**Curso:** Calidad y Preprocesamiento de Datos · Licenciatura en Ciencia de Datos · IIMAS UNAM CU · 2026‑2
**Framework:** DAMA‑DMBOK · Fase *Improve* (modelo canónico) y entrada a *Analyze*

---

## Cambio crítico respecto a `Fusion_v6` — Reasignación espacial de huérfanos

**Bug detectado en v6:** la concatenación SACMEX 2018-2021 + 2022-2024 en `Limpieza_v3` mapeó los nombres de columnas pero **no homogeneizó la normalización**. El histórico llegó a fusion con nombres crudos:
* Números de calle: `"100"`, `"11"`, `"1860"`, `"35B"`
* Andadores y cerradas: `"1a. Cerrada Amantecatl"`, `"3ER. Andador Osa Mayor"`
* Genéricos: `"Ciudad de México"` (50 K reportes), `"Mexico"`, `"Mexico City"`
* Mezclas: `"Santa Fe Cuajimalpa"` (colonia + alcaldía juntas)

Cada uno generó una "colonia huérfana" en el catálogo, inflándolo de ~1 950 a 6 936. **110 756 fugas (21 %) quedaron atribuidas a colonias fantasma con `pob_colonia = 0`** y `fugas_por_10k_hab_total = NaN` en 73.5 % de las filas.

**Solución v7 — reasignación espacial:**

> El 99 % de los reportes huérfanos tiene coordenadas válidas. Reasignamos cada uno a la
> **UT IECM cuyo polígono lo contiene** (point-in-polygon con geopandas), o si está fuera
> de todos los polígonos, a la UT cuyo centroide es más cercano (haversine).

**Resultado esperado:**
* Catálogo vuelve a ~1 837 IECM + ~30 colonias verdaderamente sin coordenadas
* ~110 K reportes recuperados, atribuidos a UT con población real
* `fugas_por_10k_hab_total` calculable para 100 % de las filas
* IVH y XGBoost sin ruido de huérfanas vacías

## Cambios mantenidos respecto a v6

* Universo IECM 2022 (1 837 UT con polígonos y población oficial)
* SACMEX 2018-2024 (~519 K reportes, 13 periodos semestrales)
* Plantas potabilizadoras + dist_planta_km
* Morbilidad estimada por factor de riesgo
* IVH con log+pesos efectivos, IVS, score_intervencion_base
* Calidad agua nearest-sitio
* SEPOMEX → cp_a_colonia para vista ciudadano

## Estructura

0. Configuración del entorno
1. Funciones reutilizables (incluye `reasignar_punto_a_polygon`)
2. Carga de datos limpios
3. Record-linkage SACMEX↔IECM (pre-cómputo de nombres)
4. Construcción de tablas maestras dimensionales
   * **4.6 incidentes_fugas con reasignación espacial de huérfanos (NUEVO en v7)**
   * **4.1 territorios excluye colonias fantasma (NUEVO en v7)**
5. Distribución a granularidad colonia
6. Tabla maestra colonia (snapshot)
7. Tabla maestra colonia × año
8. Tabla maestra colonia × semestre
9. Índices de vulnerabilidad (IVH e IVS)
10. Cruce SEPOMEX → IECM (vista ciudadano por CP)
11. Verificación post-fusión
12. Export del modelo canónico
13. Conclusiones

## 1. Configuración del entorno

In [1]:
# Instalación (descomentar primera vez):
# !pip install pandas numpy rapidfuzz scipy matplotlib seaborn geopandas

In [2]:
import os, re, json, warnings, unicodedata
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

try:
    from rapidfuzz import fuzz, process as rf_process
    HAS_RAPIDFUZZ = True
except ImportError:
    HAS_RAPIDFUZZ = False
    from difflib import SequenceMatcher

try:
    from scipy.spatial import cKDTree
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    HAS_GPD = False
    print("⚠️ geopandas no instalado; el GeoJSON de IECM no podrá leerse — instalar: pip install geopandas")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 80)

### 1.1 Raíz del proyecto y rutas

In [3]:
def get_project_root(marker: str = "README.md") -> Path:
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

PROJECT_ROOT   = get_project_root()
DATOS_LIMPIOS  = PROJECT_ROOT / "datos" / "datos_limpios"
DATOS_MAESTROS = PROJECT_ROOT / "datos" / "datos_maestros"
DATOS_MAESTROS.mkdir(parents=True, exist_ok=True)

print(f"📂 PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"📂 DATOS_LIMPIOS  : {DATOS_LIMPIOS}")
print(f"📂 DATOS_MAESTROS : {DATOS_MAESTROS}")
assert DATOS_LIMPIOS.exists()

RUTAS_IN = {
    "morbilidad":    DATOS_LIMPIOS / "morbilidad_cdmx_limpio.csv",
    "inegi":         DATOS_LIMPIOS / "inegi_cdmx_limpio.csv",
    "coneval":       DATOS_LIMPIOS / "coneval_cdmx_limpio.csv",
    "sepomex":       DATOS_LIMPIOS / "sepomex_cdmx_limpio.csv",      # secundaria (CP)
    "iecm":          DATOS_LIMPIOS / "colonias_iecm_limpio.csv",     # primaria v6
    "iecm_geojson":  DATOS_LIMPIOS / "colonias_iecm.geojson",
    "sacmex":        DATOS_LIMPIOS / "sacmex_cdmx_limpio.csv",       # ya concatenado en limpieza v3
    "conagua_sit":   DATOS_LIMPIOS / "conagua_sitios_limpio.csv",
    "conagua_res":   DATOS_LIMPIOS / "conagua_resultados_cdmx_limpio.csv",
    "alcaldias":     DATOS_LIMPIOS / "catalogo_alcaldias_cdmx.csv",
    "plantas":       PROJECT_ROOT / "datos" / "datos_crudos" / "plantas_cdmx.csv",
}

RUTAS_OUT = {
    "territorios":             DATOS_MAESTROS / "territorios_cdmx.csv",
    "sociodemografia":         DATOS_MAESTROS / "sociodemografia_alcaldia.csv",
    "morbilidad":              DATOS_MAESTROS / "morbilidad_cdmx.csv",
    "sitios_monitoreo":        DATOS_MAESTROS / "sitios_monitoreo.csv",
    "calidad_agua":            DATOS_MAESTROS / "calidad_agua.csv",
    "incidentes_fugas":        DATOS_MAESTROS / "incidentes_fugas.csv",
    "plantas_potabilizadoras": DATOS_MAESTROS / "plantas_potabilizadoras.csv",
    "maestra_colonia":         DATOS_MAESTROS / "maestra_colonia.csv",
    "maestra_colonia_anio":    DATOS_MAESTROS / "maestra_colonia_anio.csv",
    "maestra_colonia_semestre":DATOS_MAESTROS / "maestra_colonia_semestre.csv",
    "vulnerabilidad_hidrica":  DATOS_MAESTROS / "vulnerabilidad_hidrica_colonia.csv",
    "cp_a_colonia":            DATOS_MAESTROS / "cp_a_colonia.csv",  # vista ciudadano
    "log_fusion":              DATOS_MAESTROS / "_log_fusion.json",
    "linkage_huerfanos":       DATOS_MAESTROS / "_linkage_sacmex_huerfanos.csv",
}

📂 PROJECT_ROOT   : c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos
📂 DATOS_LIMPIOS  : c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos\datos\datos_limpios
📂 DATOS_MAESTROS : c:\Users\PC\Desktop\Antigravity\ProyectoFinal_CalidadDatos\datos\datos_maestros


### 1.2 Constantes globales

In [4]:
FUZZY_AUTO       = 85
FUZZY_CANDIDATO  = 70
ANIO_REFERENCIA  = 2024

NOM_127 = {
    "as_tot":         {"limite": 0.010, "param": "Arsénico"},
    "pb_tot":         {"limite": 0.010, "param": "Plomo"},
    "hg_tot":         {"limite": 0.006, "param": "Mercurio"},
    "cd_tot":         {"limite": 0.005, "param": "Cadmio"},
    "fluoruros_tot":  {"limite": 1.500, "param": "Fluoruros"},
    "n_no3":          {"limite": 10.00, "param": "Nitratos"},
    "e_coli":         {"limite": 0.000, "param": "E. coli"},
    "coli_fec":       {"limite": 0.000, "param": "Coliformes fecales"},
}

PESOS_IVH = {"pobreza":0.40, "calidad_agua":0.25, "fugas_per_capita":0.25, "morbilidad":0.10}
assert abs(sum(PESOS_IVH.values()) - 1.0) < 1e-9

LOG = defaultdict(dict)

## 2. Funciones reutilizables

In [5]:
def normalizar_texto(serie):
    return (serie.fillna("").astype(str)
            .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
            .str.upper().str.strip())


def merge_auditado(left, right, *, on, how="left", left_label="izq", right_label="der"):
    n_left, n_right = len(left), len(right)
    out = left.merge(right, on=on, how=how)
    n_out = len(out)
    inflado = (n_out > n_left) and (how in ("left", "inner"))
    flag = "⚠️" if inflado else "✅"
    print(f"  {flag} merge {how:<5} on {on}  |  {left_label}={n_left:,}  {right_label}={n_right:,}  →  out={n_out:,}")
    return out


def fuzzy_match_uno_a_uno(consultas, universo, threshold_auto=FUZZY_AUTO, threshold_cand=FUZZY_CANDIDATO):
    resultados = []
    universo_unique = list(dict.fromkeys(universo))
    for q in consultas:
        if HAS_RAPIDFUZZ:
            match = rf_process.extractOne(q, universo_unique, scorer=fuzz.token_set_ratio)
            if match is None:
                resultados.append({"consulta": q, "match": None, "score": 0, "decision": "no_match"})
                continue
            best, score, _idx = match
        else:
            mejor_score, mejor = 0, None
            for u in universo_unique:
                s = SequenceMatcher(None, q, u).ratio() * 100
                if s > mejor_score:
                    mejor_score, mejor = s, u
            best, score = mejor, int(mejor_score)
        decision = ("auto" if score >= threshold_auto
                    else "candidato" if score >= threshold_cand
                    else "no_match")
        resultados.append({"consulta": q, "match": best, "score": int(score), "decision": decision})
    return pd.DataFrame(resultados)


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def asignar_id_mas_cercano(df_a, df_b, lat_a, lon_a, lat_b, lon_b, id_b):
    a = df_a.copy(); b = df_b.dropna(subset=[lat_b, lon_b]).copy()
    lat0 = b[lat_b].mean()
    fl = np.cos(np.radians(lat0)) * 111.0
    fa = 111.0
    bxy = np.column_stack([b[lat_b].values * fa, b[lon_b].values * fl])
    axy = np.column_stack([a[lat_a].values * fa, a[lon_a].values * fl])
    if HAS_SCIPY:
        tree = cKDTree(bxy)
        _, idx = tree.query(axy, k=1)
    else:
        idx = np.empty(len(axy), dtype=int)
        for i, c in enumerate(axy):
            d = np.sqrt(((bxy - c) ** 2).sum(axis=1))
            idx[i] = np.argmin(d)
    return pd.Series(b[id_b].values[idx], index=a.index)


def header(title: str, char: str = "="):
    print(f"\n{char*70}\n {title}\n{char*70}")


def reasignar_punto_a_ut_iecm(df_reportes, gdf_iecm, lat_col="latitud", lon_col="longitud",
                                 id_col_iecm="id_colonia", buffer_km=2.0):
    """Reasigna cada reporte (con lat/lon) a la UT IECM cuyo polígono lo contiene.

    Si el punto no cae en ningún polígono (raro: bordes, fuera de CDMX), lo asigna a
    la UT cuyo centroide es más cercano (limitado por buffer_km para evitar asignaciones
    absurdas a UTs muy lejanas).

    Returns:
        Series con id_colonia asignado (NaN si no hay coords o queda fuera del buffer)
    """
    if not HAS_GPD:
        # Fallback sin geopandas: solo nearest centroide
        print("  ⚠️ Sin geopandas, usando nearest centroide (sin point-in-polygon)")
        df_iecm_pts = pd.DataFrame({
            "id_colonia": gdf_iecm["id_colonia"],
            "centroide_lat": gdf_iecm.get("centroide_lat", gdf_iecm.geometry.centroid.y if hasattr(gdf_iecm, "geometry") else None),
            "centroide_lon": gdf_iecm.get("centroide_lon", gdf_iecm.geometry.centroid.x if hasattr(gdf_iecm, "geometry") else None),
        }).dropna(subset=["centroide_lat","centroide_lon"])
        df_rep = df_reportes.dropna(subset=[lat_col, lon_col]).copy()
        ids = asignar_id_mas_cercano(df_rep, df_iecm_pts,
                                       lat_col, lon_col,
                                       "centroide_lat","centroide_lon", "id_colonia")
        out = pd.Series(np.nan, index=df_reportes.index, dtype=object)
        out.loc[df_rep.index] = ids.values
        return out

    # Con geopandas: hacer spatial join (point-in-polygon)
    df_rep = df_reportes.dropna(subset=[lat_col, lon_col]).copy()
    if len(df_rep) == 0:
        return pd.Series(np.nan, index=df_reportes.index, dtype=object)

    gdf_pts = gpd.GeoDataFrame(
        df_rep,
        geometry=gpd.points_from_xy(df_rep[lon_col], df_rep[lat_col]),
        crs="EPSG:4326"
    )
    # Asegurar mismo CRS
    if gdf_iecm.crs is None:
        gdf_iecm = gdf_iecm.set_crs("EPSG:4326")
    elif gdf_iecm.crs.to_epsg() != 4326:
        gdf_iecm = gdf_iecm.to_crs("EPSG:4326")

    # Spatial join
    joined = gpd.sjoin(gdf_pts, gdf_iecm[[id_col_iecm, "geometry"]],
                        how="left", predicate="within")
    # Resultado: index_right = índice de la UT IECM, id_colonia (sufijo si hay colisión)
    col_id = id_col_iecm + "_right" if id_col_iecm + "_right" in joined.columns else id_col_iecm
    if col_id not in joined.columns:
        # Fallback: la columna pudo quedar sin sufijo
        col_id = [c for c in joined.columns if c.startswith(id_col_iecm)][0]

    # Tomar solo el primer match si un punto cae en múltiples polígonos (overlaps)
    joined = joined[~joined.index.duplicated(keep="first")]

    asignados = joined[col_id]

    # Para los que NO cayeron en ningún polígono, usar nearest centroide
    sin_match_mask = asignados.isna()
    n_sin_match = sin_match_mask.sum()
    if n_sin_match > 0 and "centroide_lat" in gdf_iecm.columns:
        df_sin = df_rep[sin_match_mask].copy()
        df_iecm_pts = pd.DataFrame({
            "id_colonia": gdf_iecm["id_colonia"].values,
            "centroide_lat": gdf_iecm["centroide_lat"].values,
            "centroide_lon": gdf_iecm["centroide_lon"].values,
        }).dropna(subset=["centroide_lat","centroide_lon"])
        ids_near = asignar_id_mas_cercano(df_sin, df_iecm_pts,
                                            lat_col, lon_col,
                                            "centroide_lat","centroide_lon", "id_colonia")
        # Calcular distancia y filtrar > buffer_km
        df_iecm_lookup = df_iecm_pts.set_index("id_colonia")
        for idx, ut in ids_near.items():
            row = df_sin.loc[idx]
            ut_row = df_iecm_lookup.loc[ut]
            d = haversine_km(row[lat_col], row[lon_col],
                              ut_row["centroide_lat"], ut_row["centroide_lon"])
            if d > buffer_km:
                ids_near.loc[idx] = np.nan
        asignados.loc[sin_match_mask] = ids_near.values
        print(f"     {n_sin_match - asignados.isna().sum() + n_sin_match - asignados.notna().sum()} puntos sin polígono reasignados por nearest centroide")

    out = pd.Series(np.nan, index=df_reportes.index, dtype=object)
    out.loc[df_rep.index] = asignados.values
    return out


## 3. Carga de datos limpios

Diferencia clave vs v5: cargamos `colonias_iecm_limpio.csv` y el SACMEX ya concatenado (2018-2024).

In [6]:
# Carga del universo IECM (NUEVO en v6)
df_iecm = pd.read_csv(RUTAS_IN["iecm"], encoding="utf-8-sig",
                       dtype={"cve_alcaldia":str, "cveut":str, "id_colonia":str})
print(f"IECM: {df_iecm.shape}  Total UT: {len(df_iecm):,}  Suma POBL: {df_iecm['poblacion_iecm'].sum():,.0f}")

# Resto de fuentes
df_mor = pd.read_csv(RUTAS_IN["morbilidad"],  encoding="utf-8-sig", dtype={"cve_entidad": str})
df_ine = pd.read_csv(RUTAS_IN["inegi"],       encoding="utf-8-sig",
                      dtype={"cve_alcaldia": str, "mun": str})
df_con = pd.read_csv(RUTAS_IN["coneval"],     encoding="utf-8-sig", dtype={"cve_alcaldia": str})
df_sep = pd.read_csv(RUTAS_IN["sepomex"],     encoding="utf-8-sig",
                      dtype={"cve_alcaldia": str, "cve_estado": str, "codigo_postal": str})
df_sac = pd.read_csv(RUTAS_IN["sacmex"],      encoding="utf-8-sig", dtype={"cve_alcaldia": str},
                      low_memory=False, parse_dates=["fecha_registro_incidente", "fecha_reporte"])
df_sit = pd.read_csv(RUTAS_IN["conagua_sit"], encoding="utf-8-sig", low_memory=False)
df_res = pd.read_csv(RUTAS_IN["conagua_res"], encoding="utf-8-sig", low_memory=False)
df_alc = pd.read_csv(RUTAS_IN["alcaldias"],   encoding="utf-8-sig", dtype={"cve_alcaldia": str})

# Fix incondicional fecha_realizacion (heredado de v4)
if "fecha_realizacion" in df_res.columns:
    df_res = df_res.drop(columns=["fecha_realizacion"])
if "fecha_realización" in df_res.columns:
    df_res = df_res.rename(columns={"fecha_realización": "fecha_realizacion"})

# Parseo de fechas Excel (heredado v4)
def parsear_fecha_excel(serie):
    s = serie.copy()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    s_str = s.astype(str).str.strip()
    mask_vacio = s_str.isin(["", "nan", "NaN", "NaT", "None"])
    out.loc[mask_vacio] = pd.NaT
    mask_serial = (~mask_vacio) & s_str.str.match(r"^\d+(\.\d+)?$", na=False)
    if mask_serial.any():
        nums = pd.to_numeric(s_str[mask_serial], errors="coerce")
        nums_validos = nums[(nums >= 1) & (nums <= 80000)]
        if len(nums_validos) > 0:
            origen = pd.Timestamp("1899-12-30")
            fechas = origen + pd.to_timedelta(nums_validos, unit="D")
            out.loc[nums_validos.index] = fechas
    mask_pendiente = out.isna() & ~mask_vacio
    if mask_pendiente.any():
        out.loc[mask_pendiente] = pd.to_datetime(s_str[mask_pendiente], errors="coerce")
    return out

df_res["fecha_realizacion"] = parsear_fecha_excel(df_res["fecha_realizacion"])

# Plantas (opcional)
if RUTAS_IN["plantas"].exists():
    df_plantas = pd.read_csv(RUTAS_IN["plantas"], encoding="utf-8-sig", low_memory=False)
else:
    df_plantas = pd.DataFrame()

print(f"\nResumen de fuentes:")
for nombre, df in [("morbilidad",df_mor),("inegi",df_ine),("coneval",df_con),
                    ("sepomex",df_sep),("iecm",df_iecm),("sacmex",df_sac),
                    ("conagua_sit",df_sit),("conagua_res",df_res),("plantas",df_plantas)]:
    print(f"  {nombre:<14} {df.shape}")

# v7: También cargar GeoJSON con polígonos para reasignación espacial
if HAS_GPD and RUTAS_IN["iecm_geojson"].exists():
    gdf_iecm = gpd.read_file(RUTAS_IN["iecm_geojson"])
    # Asegurar que tenemos centroide_lat/lon disponibles
    if "centroide_lat" not in gdf_iecm.columns:
        gdf_iecm["centroide_lat"] = gdf_iecm.geometry.centroid.y
        gdf_iecm["centroide_lon"] = gdf_iecm.geometry.centroid.x
    print(f"GeoJSON IECM cargado: {len(gdf_iecm)} polígonos, CRS={gdf_iecm.crs}")
else:
    gdf_iecm = None
    if not HAS_GPD:
        print("⚠️ geopandas no disponible; reasignación espacial usará solo nearest centroide")
    else:
        print(f"⚠️ {RUTAS_IN['iecm_geojson'].name} no encontrado")


IECM: (1837, 15)  Total UT: 1,837  Suma POBL: 9,209,944

Resumen de fuentes:
  morbilidad     (8, 33)
  inegi          (494, 25)
  coneval        (190, 12)
  sepomex        (1526, 8)
  iecm           (1837, 15)
  sacmex         (519033, 18)
  conagua_sit    (7756, 16)
  conagua_res    (196, 96)
  plantas        (45, 7)
GeoJSON IECM cargado: 1837 polígonos, CRS=EPSG:4326


### 3.1 Verificación de cobertura por alcaldía

In [7]:
matriz_cob = pd.DataFrame(index=df_alc["cve_alcaldia"])
matriz_cob["nombre"]  = df_alc["nombre_oficial"].values
matriz_cob["INEGI"]   = matriz_cob.index.isin(df_ine["cve_alcaldia"].dropna()).astype(int)
matriz_cob["CONEVAL"] = matriz_cob.index.isin(df_con["cve_alcaldia"].dropna()).astype(int)
matriz_cob["SEPOMEX"] = matriz_cob.index.isin(df_sep["cve_alcaldia"].dropna()).astype(int)
matriz_cob["IECM"]    = matriz_cob.index.isin(df_iecm["cve_alcaldia"].dropna()).astype(int)
matriz_cob["SACMEX"]  = matriz_cob.index.isin(df_sac["cve_alcaldia"].dropna()).astype(int)
matriz_cob["cobertura"] = matriz_cob[["INEGI","CONEVAL","SEPOMEX","IECM","SACMEX"]].sum(axis=1)
display(matriz_cob)
n_completas = (matriz_cob["cobertura"] == 5).sum()
print(f"\nAlcaldías con cobertura completa (5/5): {n_completas} de 16")

,nombre,INEGI,CONEVAL,SEPOMEX,IECM,SACMEX,cobertura
cve_alcaldia,,,,,,,
002,Azcapotzalco,1,1,1,1,1,5
003,Coyoacán,1,1,1,1,1,5
004,Cuajimalpa de Morelos,1,1,1,1,1,5
005,Gustavo A. Madero,1,1,1,1,1,5
006,Iztacalco,1,1,1,1,1,5
007,Iztapalapa,1,1,1,1,1,5
008,La Magdalena Contreras,1,1,1,1,1,5
009,Milpa Alta,1,1,1,1,1,5
010,Álvaro Obregón,1,1,1,1,1,5



Alcaldías con cobertura completa (5/5): 16 de 16


## 3.2 Record-linkage SACMEX ↔ IECM

Cambio crítico vs v5: el linkage ahora se hace contra IECM (1 837 UT) en lugar de SEPOMEX (1 503 colonias). El match exacto sube de ~25 % a 39.5 % y con fuzzy esperamos llegar a ~85 %.

In [8]:
header("3.2 Record-linkage SACMEX ↔ IECM")

# Pares únicos en SACMEX y en IECM
pares_sac = df_sac[["cve_alcaldia", "colonia_norm"]].drop_duplicates().dropna()
pares_iecm = df_iecm[["cve_alcaldia", "nom_colonia_norm"]].drop_duplicates().dropna() \
              .rename(columns={"nom_colonia_norm": "colonia_norm"})

mi_iecm = pd.MultiIndex.from_frame(pares_iecm[["cve_alcaldia","colonia_norm"]])
mi_sac_p = pd.MultiIndex.from_frame(pares_sac[["cve_alcaldia","colonia_norm"]])
mask_huerf = ~mi_sac_p.isin(mi_iecm)
huerfanos = pares_sac[mask_huerf]

print(f"  Pares SACMEX únicos: {len(pares_sac):,}")
print(f"  Match exacto IECM:   {(~mask_huerf).sum():,} ({100*(~mask_huerf).mean():.1f}%)")
print(f"  Huérfanos SACMEX:    {len(huerfanos):,}")

# Fuzzy match dentro de cada alcaldía
filas_linkage = []
for cve, sub in huerfanos.groupby("cve_alcaldia"):
    universo_alc = pares_iecm.loc[pares_iecm["cve_alcaldia"] == cve, "colonia_norm"].tolist()
    consultas = sub["colonia_norm"].tolist()
    if not universo_alc:
        for q in consultas:
            filas_linkage.append({"cve_alcaldia": cve, "consulta": q,
                                   "match": None, "score": 0, "decision": "no_match"})
        continue
    df_match = fuzzy_match_uno_a_uno(consultas, universo_alc)
    df_match["cve_alcaldia"] = cve
    filas_linkage.append(df_match)

linkage = pd.concat(filas_linkage, ignore_index=True) if filas_linkage else pd.DataFrame()
print("\n  Resumen del fuzzy matching:")
print(linkage["decision"].value_counts())

# Construir el mapa de linkage
MAPA_LINKAGE = (linkage[linkage["decision"]=="auto"]
                .set_index(["cve_alcaldia","consulta"])["match"].to_dict())
print(f"\n  Mapa linkage: {len(MAPA_LINKAGE)} entradas")

# Exportar candidatos para revisión manual
candidatos = linkage[linkage["decision"].isin(["candidato","no_match"])].copy()
candidatos.to_csv(RUTAS_OUT["linkage_huerfanos"], index=False, encoding="utf-8-sig")
print(f"  💾 {RUTAS_OUT['linkage_huerfanos'].name} ({len(candidatos)} candidatos)")

# Calcular cobertura final post-linkage
n_resueltos = (linkage["decision"]=="auto").sum()
cobertura_final = (~mask_huerf).sum() + n_resueltos
print(f"\n  Cobertura final SACMEX→IECM: {cobertura_final:,}/{len(pares_sac):,} ({100*cobertura_final/len(pares_sac):.1f}%)")


 3.2 Record-linkage SACMEX ↔ IECM
  Pares SACMEX únicos: 7,294
  Match exacto IECM:   740 (10.1%)
  Huérfanos SACMEX:    6,554

  Resumen del fuzzy matching:
decision
no_match     4483
auto         1455
candidato     616
Name: count, dtype: int64

  Mapa linkage: 1455 entradas
  💾 _linkage_sacmex_huerfanos.csv (5099 candidatos)

  Cobertura final SACMEX→IECM: 2,195/7,294 (30.1%)


## 4. Tablas maestras dimensionales

### 4.1 `territorios_cdmx` desde IECM (NUEVO en v6)

El catálogo principal ahora viene del IECM: 1 837 UT con polígonos, población oficial y centroides geométricos. Las huérfanas SACMEX **no resueltas** por fuzzy se agregan como `origen='sacmex_no_oficial'` (igual que en v5 pero con menor número porque IECM tiene más cobertura).

In [9]:
header("4.1 territorios_cdmx desde IECM (v7: solo huérfanas verdaderas)")

# Base IECM (1 837 UT con polígonos y población oficial)
t_iecm = df_iecm[["id_colonia","cveut","cve_alcaldia","nom_alcaldia","nom_colonia",
                   "nom_colonia_norm","poblacion_iecm","centroide_lat","centroide_lon"]].copy()
t_iecm["origen"] = "iecm"
t_iecm["codigos_postales"] = ""
t_iecm["centroide_imputado"] = 0
print(f"  Universo IECM: {len(t_iecm):,} UT")

# === v7: Las huérfanas SE LIMITAN a las que tras el linkage Y la reasignación espacial
# todavía NO encuentran UT. Es decir: nombres no resueltos por fuzzy (113 según linkage)
# que ADEMÁS no fueron reasignados espacialmente. Esto se evalúa SOBRE incidentes_master.
# Como territorios se construye ANTES que incidentes (la sección 4.6 lo procesa después),
# usamos la lógica del fuzzy linkage como aproximación: solo entran las que fuzzy NO
# resolvió. La reasignación espacial las sacará del universo después (algunas pueden
# quedar sin id_colonia si no tienen coords; para esas creamos un id sintético mínimo).

huerfanas_no_resueltas = linkage[linkage["decision"]!="auto"][["cve_alcaldia","consulta"]].copy()
huerfanas_no_resueltas = huerfanas_no_resueltas.rename(columns={"consulta":"colonia_norm"})

pares_sac_unicos = (df_sac.dropna(subset=["cve_alcaldia","colonia_norm","colonia_catalogo"])
                          [["cve_alcaldia","colonia_catalogo","colonia_norm"]]
                          .drop_duplicates())
mi_no_res = pd.MultiIndex.from_frame(huerfanas_no_resueltas[["cve_alcaldia","colonia_norm"]])
mi_phr = pd.MultiIndex.from_frame(pares_sac_unicos[["cve_alcaldia","colonia_norm"]])
mask_no_resuelta = mi_phr.isin(mi_no_res)
t_huerf = pares_sac_unicos[mask_no_resuelta].copy()
t_huerf = (t_huerf.groupby(["cve_alcaldia","colonia_norm"], as_index=False)
                  .agg(nom_colonia=("colonia_catalogo","first")))
t_huerf = t_huerf.rename(columns={"colonia_norm":"nom_colonia_norm"})
t_huerf = t_huerf.merge(df_alc[["cve_alcaldia","nombre_oficial"]],
                         on="cve_alcaldia", how="left").rename(columns={"nombre_oficial":"nom_alcaldia"})
t_huerf["origen"] = "sacmex_no_oficial"
t_huerf["codigos_postales"] = ""
t_huerf["poblacion_iecm"] = 0
t_huerf["centroide_lat"] = np.nan
t_huerf["centroide_lon"] = np.nan
t_huerf["cveut"] = "huerf_" + t_huerf["cve_alcaldia"] + "_" + t_huerf.index.astype(str)
t_huerf["id_colonia"] = t_huerf["cveut"]

# Imputar centroide a huérfanas
centroides_alc = (t_iecm.groupby("cve_alcaldia", as_index=False)
                  .agg(c_lat=("centroide_lat","median"), c_lon=("centroide_lon","median")))
t_huerf = t_huerf.merge(centroides_alc, on="cve_alcaldia", how="left")
t_huerf["centroide_lat"] = t_huerf["centroide_lat"].fillna(t_huerf["c_lat"])
t_huerf["centroide_lon"] = t_huerf["centroide_lon"].fillna(t_huerf["c_lon"])
t_huerf = t_huerf.drop(columns=["c_lat","c_lon"])
t_huerf["centroide_imputado"] = 1
print(f"  Huérfanas SACMEX no resueltas por fuzzy: {len(t_huerf):,}")

# Unir
COLS_ORDEN = ["id_colonia","cveut","cve_alcaldia","nom_alcaldia","nom_colonia",
              "nom_colonia_norm","codigos_postales","origen",
              "poblacion_iecm","centroide_lat","centroide_lon","centroide_imputado"]
territorios = pd.concat([t_iecm[COLS_ORDEN], t_huerf[COLS_ORDEN]], ignore_index=True)

n_dup = territorios.duplicated(subset=["id_colonia"]).sum()
if n_dup > 0:
    territorios = territorios.drop_duplicates(subset=["id_colonia"], keep="first")

print(f"\n  Catálogo final v7 (preliminar): {len(territorios):,} colonias")
print(f"  Distribución por origen:")
print(territorios["origen"].value_counts())
print(f"\n  📌 Tras la reasignación espacial en sección 4.6, las huérfanas que SE USAN")
print(f"     se reducen aún más (las que se reasignan a IECM NO se cuentan dos veces).")

LOG["territorios_cdmx"] = {
    "n_filas": len(territorios),
    "n_iecm": int((territorios["origen"]=="iecm").sum()),
    "n_no_oficial": int((territorios["origen"]=="sacmex_no_oficial").sum()),
}


 4.1 territorios_cdmx desde IECM (v7: solo huérfanas verdaderas)
  Universo IECM: 1,837 UT
  Huérfanas SACMEX no resueltas por fuzzy: 5,099

  Catálogo final v7 (preliminar): 6,936 colonias
  Distribución por origen:
origen
sacmex_no_oficial    5099
iecm                 1837
Name: count, dtype: int64

  📌 Tras la reasignación espacial en sección 4.6, las huérfanas que SE USAN
     se reducen aún más (las que se reasignan a IECM NO se cuentan dos veces).


### 4.2 `sociodemografia_alcaldia` (sin cambios mayores vs v5)

In [10]:
header("4.2 sociodemografia_alcaldia")

COLS_AGG_INEGI = ["pobtot","pob0_14","pob15_64","pob65_mas",
                  "vivtot","tvivparhab","vph_aguadv","vph_aguafv",
                  "vph_tinaco","vph_cister","vph_drenaj","vph_nodren",
                  "vph_c_serv","vph_excsa"]
agg_inegi = (df_ine.groupby("cve_alcaldia", as_index=False)[COLS_AGG_INEGI].sum())
agg_inegi = agg_inegi.rename(columns={"pobtot":"pob_inegi","vivtot":"viviendas_inegi"})
agg_inegi["pct_aguadv"]  = (agg_inegi["vph_aguadv"]  / agg_inegi["tvivparhab"] * 100).round(2)
agg_inegi["pct_drenaje"] = (agg_inegi["vph_drenaj"]  / agg_inegi["tvivparhab"] * 100).round(2)
agg_inegi["pct_tinaco"]  = (agg_inegi["vph_tinaco"]  / agg_inegi["tvivparhab"] * 100).round(2)

# IECM por alcaldía (NUEVO: agregar también pob IECM)
agg_iecm = (df_iecm.groupby("cve_alcaldia", as_index=False)
            .agg(pob_iecm=("poblacion_iecm","sum"), n_ut=("id_colonia","count")))

df_con_2020 = df_con[df_con["anio"] == df_con["anio"].max()].copy()
agg_coneval = (df_con_2020.groupby("cve_alcaldia", as_index=False)
               .agg(pob_coneval=("poblacion","sum"),
                    pobreza_pct_promedio=("pobreza_porcentaje","mean"),
                    ingreso_lpi_pct_promedio=("ingreso_inferior_a_lpi_porcentaje","mean"),
                    carencia_calidad_espacios_pct=("carencia_calidad_espacios_vivienda_porcentaje","mean")))
for c in ["pobreza_pct_promedio","ingreso_lpi_pct_promedio","carencia_calidad_espacios_pct"]:
    agg_coneval[c] = agg_coneval[c].round(2)

sociodemo = merge_auditado(df_alc[["cve_alcaldia","nombre_oficial"]], agg_inegi,
                           on="cve_alcaldia", how="left",
                           left_label="catalogo", right_label="inegi")
sociodemo = merge_auditado(sociodemo, agg_iecm, on="cve_alcaldia", how="left",
                           left_label="prev", right_label="iecm")
sociodemo = merge_auditado(sociodemo, agg_coneval, on="cve_alcaldia", how="left",
                           left_label="prev", right_label="coneval")
sociodemo["err_pob_inegi_iecm_pct"] = ((sociodemo["pob_inegi"] - sociodemo["pob_iecm"]).abs()
                                         / sociodemo["pob_inegi"] * 100).round(2)

display(sociodemo[["nombre_oficial","pob_inegi","pob_iecm","pob_coneval",
                    "err_pob_inegi_iecm_pct","pobreza_pct_promedio","pct_aguadv"]])
LOG["sociodemografia_alcaldia"] = {"n_filas": len(sociodemo)}


 4.2 sociodemografia_alcaldia
  ✅ merge left  on cve_alcaldia  |  catalogo=16  inegi=16  →  out=16
  ✅ merge left  on cve_alcaldia  |  prev=16  iecm=16  →  out=16
  ✅ merge left  on cve_alcaldia  |  prev=16  coneval=16  →  out=16


,nombre_oficial,pob_inegi,pob_iecm,pob_coneval,err_pob_inegi_iecm_pct,pobreza_pct_promedio,pct_aguadv
0,Azcapotzalco,432205,432205,404958,0.00,24.59,99.57
1,Coyoacán,614447,614447,568912,0.00,27.50,99.76
2,Cuajimalpa de Morelos,217643,214050,199873,1.65,31.77,99.34
3,Gustavo A. Madero,1173351,1173351,1130265,0.00,33.69,99.63
4,Iztacalco,404695,404643,376977,0.01,25.72,99.69
5,Iztapalapa,1835486,1838464,1760382,0.16,42.87,99.59
6,La Magdalena Contreras,247580,247791,241691,0.09,42.17,98.68
7,Milpa Alta,152538,143976,161267,5.61,53.44,89.37
8,Álvaro Obregón,759133,762909,726045,0.50,37.67,99.54
9,Tláhuac,392253,382901,377745,2.38,41.50,97.60


### 4.3 `morbilidad_cdmx` (sin cambios)

In [11]:
header("4.3 morbilidad_cdmx")

morbi = df_mor.copy()
pob_cdmx = sociodemo["pob_inegi"].sum()
morbi["tasa_por_100k"] = (morbi["acumulado"] / pob_cdmx * 100_000).round(2)

COLS_MORBI = ["cve_diagno","des_diagno","periodo","acumulado","tasa_por_100k","totalm","totalf",
              "menores_1","de01_a_04","de05_a_09","de10_a_14","de15_a_19","de20_a_24",
              "de25_a_44","de45_a_49","de50_a_59","de60_a_64","de65_y_mas"]
morbi = morbi[COLS_MORBI].sort_values("acumulado", ascending=False).reset_index(drop=True)

TASA_MORBI_ESTATAL = float(morbi["tasa_por_100k"].sum())
CASOS_MORBI_ESTATAL = int(morbi["acumulado"].sum())
display(morbi[["des_diagno","acumulado","tasa_por_100k"]])
LOG["morbilidad_cdmx"] = {"n_filas": len(morbi),
                            "tasa_estatal": round(TASA_MORBI_ESTATAL, 2)}


 4.3 morbilidad_cdmx


,des_diagno,acumulado,tasa_por_100k
0,Infecciones intestinales por otros organismos ...,449762,4883.80
1,Amebiasis intestinal,6621,71.90
2,Otras infecciones intestinales debidas a proto...,3534,38.37
3,Giardiasis,508,5.52
4,Fiebre tifoidea,156,1.69
5,Shigelosis,38,0.41
6,Fiebre paratifoidea,27,0.29
7,Cólera,0,0.00


### 4.4 `sitios_monitoreo` y 4.5 `calidad_agua` (sin cambios mayores vs v5)

In [12]:
header("4.4 sitios_monitoreo + 4.5 calidad_agua")

clasif = (df_res[["clave_sitio","clasificacion_cdmx"]].drop_duplicates()
          .dropna(subset=["clasificacion_cdmx"]).drop_duplicates(subset=["clave_sitio"]))
sitios_master = df_sit[df_sit["clave_sitio"].isin(clasif["clave_sitio"])].copy()
sitios_master["latitud"]  = pd.to_numeric(sitios_master["latitud"], errors="coerce")
sitios_master["longitud"] = pd.to_numeric(sitios_master["longitud"], errors="coerce")
sitios_master = sitios_master.merge(clasif, on="clave_sitio", how="left")
print(f"  Sitios CONAGUA en scope: {len(sitios_master)}")
print(sitios_master.groupby("clasificacion_cdmx").size())

# Calidad: flags NOM-127
calidad = df_res.copy()
calidad["anio_medicion"] = calidad["fecha_realizacion"].dt.year.fillna(calidad["año"]).astype("Int64")
flags_nom = []
for col, info in NOM_127.items():
    if col not in calidad.columns: continue
    flag_col = f"excede_nom_{col}"
    calidad[flag_col] = (pd.to_numeric(calidad[col], errors="coerce") > info["limite"]).astype("Int64")
    flags_nom.append(flag_col)
calidad["excede_alguna_nom"] = (calidad[flags_nom].sum(axis=1) > 0).astype(int) if flags_nom else 0

LOG["sitios_monitoreo"] = {"n_filas": len(sitios_master)}
LOG["calidad_agua"] = {"n_filas": len(calidad), "n_excede": int(calidad["excede_alguna_nom"].sum())}


 4.4 sitios_monitoreo + 4.5 calidad_agua
  Sitios CONAGUA en scope: 24
clasificacion_cdmx
cdmx_local          20
cutzamala_rio        2
cutzamala_tuxpan     2
dtype: int64


### 4.6 `incidentes_fugas` con reasignación espacial (CORREGIDO en v7)

Aplicamos el `MAPA_LINKAGE` a los nombres reconocibles. Para los reportes que aún quedan como huérfanos (mayormente del SACMEX histórico 2018-2021 con nombres mal capturados), aplicamos **reasignación espacial**: cada reporte con coordenadas se asigna a la UT IECM cuyo polígono lo contiene (point-in-polygon vía `geopandas`); si está fuera de todos los polígonos, al centroide más cercano dentro de un buffer de 2 km.

Esto recupera ~110 K reportes que en v6 quedaban atribuidos a colonias fantasma con pob=0.

In [13]:
header("4.6 incidentes_fugas + reasignación espacial")

incidentes = df_sac.copy()
incidentes["origen_colonia"] = "match_exacto"

mi_iecm_full = pd.MultiIndex.from_frame(pares_iecm[["cve_alcaldia","colonia_norm"]])
mi_sac_full = pd.MultiIndex.from_frame(incidentes[["cve_alcaldia","colonia_norm"]])
mask_huerf_full = ~mi_sac_full.isin(mi_iecm_full) & incidentes["colonia_norm"].notna()
incidentes.loc[mask_huerf_full, "origen_colonia"] = "huerfano"

# Aplicar mapa de linkage (resuelve huérfanos por nombre similar)
def _map(row):
    if row["origen_colonia"] != "huerfano":
        return row["colonia_norm"], row["origen_colonia"]
    key = (row["cve_alcaldia"], row["colonia_norm"])
    if key in MAPA_LINKAGE:
        return MAPA_LINKAGE[key], "linkage_auto"
    return row["colonia_norm"], "huerfano"

resultado = incidentes.apply(_map, axis=1, result_type="expand")
incidentes["colonia_canonica"] = resultado[0]
incidentes["origen_colonia"]   = resultado[1]

# Resolver id_colonia para los que matchean a IECM (exacto o fuzzy)
mapa_iecm_id = df_iecm.set_index(["cve_alcaldia","nom_colonia_norm"])["id_colonia"].to_dict()

def _resolve_id(row):
    if row["origen_colonia"] == "huerfano":
        return None
    key = (row["cve_alcaldia"], row["colonia_canonica"])
    return mapa_iecm_id.get(key, None)

incidentes["id_colonia"] = incidentes.apply(_resolve_id, axis=1)

print("\nDistribución origen_colonia (antes de reasignación espacial):")
print(incidentes["origen_colonia"].value_counts())
print(f"  Reportes con id_colonia asignado: {incidentes['id_colonia'].notna().sum():,}")
print(f"  Reportes huérfanos (a reasignar): {(incidentes['origen_colonia']=='huerfano').sum():,}")

# === REASIGNACIÓN ESPACIAL DE HUÉRFANOS (NUEVO en v7) ===
header("4.6b Reasignación espacial de huérfanos")

mask_huerf_to_reasign = (incidentes["origen_colonia"]=="huerfano") & \
                          incidentes["latitud"].notna() & \
                          incidentes["longitud"].notna()
n_a_reasign = mask_huerf_to_reasign.sum()
print(f"  Huérfanos con coordenadas válidas: {n_a_reasign:,}")

if n_a_reasign > 0 and gdf_iecm is not None:
    df_huerf = incidentes[mask_huerf_to_reasign].copy()
    nuevos_ids = reasignar_punto_a_ut_iecm(
        df_huerf, gdf_iecm,
        lat_col="latitud", lon_col="longitud",
        id_col_iecm="id_colonia", buffer_km=2.0
    )
    n_reasignados = nuevos_ids.notna().sum()
    print(f"  Reasignados exitosamente:        {n_reasignados:,} ({100*n_reasignados/n_a_reasign:.1f}%)")
    print(f"  Sin polígono cercano (buffer 2km): {n_a_reasign - n_reasignados:,}")

    # Aplicar reasignación
    incidentes.loc[mask_huerf_to_reasign, "id_colonia"] = nuevos_ids.values
    incidentes.loc[mask_huerf_to_reasign & nuevos_ids.notna().reindex(incidentes.index, fill_value=False),
                    "origen_colonia"] = "reasignacion_espacial"

    # Para los reasignados, actualizar colonia_canonica también desde el catálogo IECM
    iecm_lookup = df_iecm.set_index("id_colonia")["nom_colonia_norm"].to_dict()
    mask_reasignado = (incidentes["origen_colonia"]=="reasignacion_espacial")
    incidentes.loc[mask_reasignado, "colonia_canonica"] = \
        incidentes.loc[mask_reasignado, "id_colonia"].map(iecm_lookup)
elif gdf_iecm is None:
    print("  ⚠️ Sin gdf_iecm; reasignación espacial saltada. Los huérfanos seguirán como tal.")

# Distribución final
print("\nDistribución origen_colonia (FINAL v7):")
print(incidentes["origen_colonia"].value_counts())
n_con_id = incidentes["id_colonia"].notna().sum()
print(f"\nReportes con id_colonia: {n_con_id:,}/{len(incidentes):,} ({100*n_con_id/len(incidentes):.1f}%)")

# Subset
COLS_INC = ["folio_incidente","fecha_registro_incidente","fecha_reporte",
            "lag_dias","flag_lag_largo","clasificacion","reporte","medio_recepcion",
            "alcaldia_catalogo","cve_alcaldia","colonia_catalogo","colonia_canonica",
            "id_colonia","origen_colonia","latitud","longitud"]
COLS_INC = [c for c in COLS_INC if c in incidentes.columns]
incidentes_master = incidentes[COLS_INC].copy()
incidentes_master["anio_reporte"] = pd.to_datetime(incidentes_master["fecha_reporte"], errors="coerce").dt.year

LOG["incidentes_fugas"] = {
    "n_filas": len(incidentes_master),
    "n_con_id_colonia": int(n_con_id),
    "origen_colonia_dist": incidentes_master["origen_colonia"].value_counts().to_dict(),
    "n_reasignados_espacialmente": int((incidentes_master["origen_colonia"]=="reasignacion_espacial").sum()),
}


 4.6 incidentes_fugas + reasignación espacial

Distribución origen_colonia (antes de reasignación espacial):
origen_colonia
linkage_auto    250421
match_exacto    154893
huerfano        113719
Name: count, dtype: int64
  Reportes con id_colonia asignado: 400,115
  Reportes huérfanos (a reasignar): 113,719

 4.6b Reasignación espacial de huérfanos
  Huérfanos con coordenadas válidas: 113,719
     -110157 puntos sin polígono reasignados por nearest centroide
  Reasignados exitosamente:        112,486 (98.9%)
  Sin polígono cercano (buffer 2km): 1,233

Distribución origen_colonia (FINAL v7):
origen_colonia
linkage_auto             250421
match_exacto             154893
reasignacion_espacial    112486
huerfano                   1233
Name: count, dtype: int64

Reportes con id_colonia: 512,601/519,033 (98.8%)


### 4.7 `plantas_potabilizadoras` (sin cambios vs v5)

In [14]:
header("4.7 plantas_potabilizadoras")

if len(df_plantas) == 0:
    print("  ⚠️ Sin plantas; saltamos esta tabla")
    plantas_master = pd.DataFrame()
else:
    plantas_master = df_plantas.copy()
    plantas_master.columns = [c.lower().strip().replace(" ","_") for c in plantas_master.columns]

    col_municipio = next((c for c in plantas_master.columns
                          if "municipio" in c or "alcald" in c or "delegacion" in c), None)
    col_nombre = next((c for c in plantas_master.columns
                        if "nombre" in c or "planta" in c), plantas_master.columns[0])

    if col_municipio:
        plantas_master["municipio_norm"] = normalizar_texto(plantas_master[col_municipio])
        df_alc_norm = df_alc.copy()
        df_alc_norm["nombre_norm"] = normalizar_texto(df_alc_norm["nombre_oficial"])
        plantas_master = plantas_master.merge(
            df_alc_norm[["nombre_norm","cve_alcaldia"]].rename(columns={"nombre_norm":"municipio_norm"}),
            on="municipio_norm", how="left").drop(columns=["municipio_norm"])

    plantas_master["id_planta"] = "PLANTA_" + plantas_master.index.astype(str).str.zfill(3)
    if "latitud" not in plantas_master.columns:
        plantas_master["latitud"] = np.nan
        plantas_master["longitud"] = np.nan
    plantas_master["latitud"]  = pd.to_numeric(plantas_master["latitud"], errors="coerce")
    plantas_master["longitud"] = pd.to_numeric(plantas_master["longitud"], errors="coerce")

    centroides_alc_p = (territorios.groupby("cve_alcaldia", as_index=False)
                        .agg(c_lat=("centroide_lat","median"), c_lon=("centroide_lon","median")))
    plantas_master = plantas_master.merge(centroides_alc_p, on="cve_alcaldia", how="left")
    mask_sin = plantas_master["latitud"].isna()
    plantas_master.loc[mask_sin, "latitud"]  = plantas_master.loc[mask_sin, "c_lat"]
    plantas_master.loc[mask_sin, "longitud"] = plantas_master.loc[mask_sin, "c_lon"]
    plantas_master["origen_coords"] = np.where(mask_sin, "centroide_alcaldia", "directa_pdf")
    plantas_master = plantas_master.drop(columns=["c_lat","c_lon"])
    plantas_master = plantas_master.rename(columns={col_nombre:"nombre"})

    print(f"  Plantas: {len(plantas_master)}")
    print(f"  Con coords directas: {(plantas_master['origen_coords']=='directa_pdf').sum()}")

LOG["plantas_potabilizadoras"] = {"n_filas": len(plantas_master)}


 4.7 plantas_potabilizadoras
  Plantas: 45
  Con coords directas: 0


### 4.8 Compactación de territorios post-reasignación (NUEVO en v7)

Tras la reasignación espacial en 4.6, muchas huérfanas (que sólo existían porque sus reportes no encontraban UT) ya no tienen reportes asociados (sus reportes ahora se asignaron a UTs IECM). Removemos del catálogo `territorios_cdmx` aquellas huérfanas que **no son referenciadas por ningún reporte**, manteniendo solo las que sí están en uso.

In [15]:
header("4.8 Compactación de territorios post-reasignación")

ids_en_uso = set(incidentes_master["id_colonia"].dropna().unique())
ids_iecm = set(territorios.loc[territorios["origen"]=="iecm", "id_colonia"])
# Mantener: todos los IECM (siempre, incluso sin reportes) + huérfanas con reportes
n_antes = len(territorios)
mantener = (territorios["origen"]=="iecm") | (territorios["id_colonia"].isin(ids_en_uso))
territorios = territorios[mantener].reset_index(drop=True)
n_despues = len(territorios)
print(f"  Catálogo antes:    {n_antes:,}")
print(f"  Catálogo después:  {n_despues:,}")
print(f"  Removidas:         {n_antes - n_despues:,} (huérfanas sin reportes asociados tras reasignación)")
print(f"\n  Distribución final:")
print(territorios["origen"].value_counts())
LOG["territorios_cdmx"]["n_filas_v7"] = len(territorios)
LOG["territorios_cdmx"]["n_removidas_post_reasignacion"] = n_antes - n_despues


 4.8 Compactación de territorios post-reasignación
  Catálogo antes:    6,936
  Catálogo después:  1,837
  Removidas:         5,099 (huérfanas sin reportes asociados tras reasignación)

  Distribución final:
origen
iecm    1837
Name: count, dtype: int64


## 5. Distribución a granularidad colonia

Cambio clave vs v5: la **población viene de IECM** (oficial, no distribución uniforme). El resto de pobreza, demografía, calidad agua, morbilidad mantiene la misma lógica (uniforme dentro de alcaldía / nearest-sitio / factor de riesgo).

### 5.1 Demografía e infraestructura → colonia

`pob_colonia` ahora es directamente `poblacion_iecm` (oficial). Las métricas de cobertura hídrica e infraestructura siguen siendo por alcaldía (INEGI las publica así); las distribuimos por peso uniforme.

In [16]:
header("5.1 Demografía + pobreza → colonia")

# Métricas a heredar de la alcaldía (% de cobertura, pobreza, etc.)
COLS_HEREDADAS = ["pct_aguadv","pct_drenaje","pct_tinaco",
                   "pobreza_pct_promedio","ingreso_lpi_pct_promedio",
                   "carencia_calidad_espacios_pct"]

colonias_demo = territorios.merge(
    sociodemo[["cve_alcaldia"] + COLS_HEREDADAS],
    on="cve_alcaldia", how="left"
)

# pob_colonia = poblacion_iecm directamente (NO distribución uniforme — gran cambio v6)
colonias_demo["pob_colonia"] = colonias_demo["poblacion_iecm"]
colonias_demo["pob_origen"] = np.where(
    colonias_demo["origen"]=="iecm", "iecm_oficial", "huerfano_sin_pob"
)
colonias_demo["pobreza_origen"] = "distribuida_uniforme_alcaldia"

# Densidad poblacional aproximada (necesitaríamos el área del polígono IECM; aquí usamos
# el dato de alcaldía como fallback)
densidad_alc = sociodemo[["cve_alcaldia"]].copy()
densidad_alc["densidad_pob_por_vivienda"] = (sociodemo["pob_inegi"] / sociodemo["tvivparhab"]).round(2)
colonias_demo = colonias_demo.merge(densidad_alc, on="cve_alcaldia", how="left")

print(f"  Colonias con pob oficial IECM: {(colonias_demo['pob_origen']=='iecm_oficial').sum():,}")
print(f"  Colonias huérfanas sin población: {(colonias_demo['pob_origen']=='huerfano_sin_pob').sum()}")
display(colonias_demo[["id_colonia","nom_alcaldia","nom_colonia","pob_colonia",
                         "pct_aguadv","pobreza_pct_promedio"]].head())


 5.1 Demografía + pobreza → colonia
  Colonias con pob oficial IECM: 1,837
  Colonias huérfanas sin población: 0


,id_colonia,nom_alcaldia,nom_colonia,pob_colonia,pct_aguadv,pobreza_pct_promedio
0,10-001,ALVARO OBREGON,ABRAHAM GONZALEZ,459,99.54,37.67
1,10-002,ALVARO OBREGON,ACUEDUCTO,3188,99.54,37.67
2,10-003,ALVARO OBREGON,ACUILOTLA,1927,99.54,37.67
3,10-004,ALVARO OBREGON,HACIENDA DE GUADALUPE CHIMALISTAC,1257,99.54,37.67
4,10-005,ALVARO OBREGON,AGUILAS 3ER PARQUE,3832,99.54,37.67


### 5.2 Calidad de agua → colonia (KNN nearest)

In [17]:
header("5.2 Calidad agua → colonia (nearest sitio)")

sitios_local = (sitios_master[(sitios_master["clasificacion_cdmx"]=="cdmx_local")]
                .dropna(subset=["latitud","longitud"]).copy())
metrica_sitio = (calidad[calidad["clasificacion_cdmx"]=="cdmx_local"]
                 .groupby("clave_sitio", as_index=False)
                 .agg(n_mediciones_sitio=("excede_alguna_nom","size"),
                      pct_excede_nom_local=("excede_alguna_nom",
                                              lambda s: round(s.mean()*100, 1))))
sitios_local = sitios_local.merge(metrica_sitio, on="clave_sitio", how="left")

ids_cerc = asignar_id_mas_cercano(territorios, sitios_local,
                                     "centroide_lat","centroide_lon",
                                     "latitud","longitud","clave_sitio")
territorios_calidad = territorios[["id_colonia","centroide_lat","centroide_lon"]].copy()
territorios_calidad["sitio_calidad_cercano"] = ids_cerc.values
territorios_calidad = territorios_calidad.merge(
    sitios_local[["clave_sitio","latitud","longitud","pct_excede_nom_local","n_mediciones_sitio"]],
    left_on="sitio_calidad_cercano", right_on="clave_sitio", how="left"
)
territorios_calidad["dist_sitio_km"] = haversine_km(
    territorios_calidad["centroide_lat"].values,
    territorios_calidad["centroide_lon"].values,
    territorios_calidad["latitud"].values,
    territorios_calidad["longitud"].values).round(2)
territorios_calidad = territorios_calidad.drop(columns=["clave_sitio","latitud","longitud"])
territorios_calidad["calidad_origen"] = "nearest_sitio_cdmx_local"

print(f"  Distancia a sitio (km):")
print(territorios_calidad["dist_sitio_km"].describe(percentiles=[.5,.9]).round(2).to_string())


 5.2 Calidad agua → colonia (nearest sitio)
  Distancia a sitio (km):
count    1837.00
mean        5.87
std         3.20
min         0.20
50%         5.92
90%         9.98
max        17.12


### 5.3 Calidad sistémica + morbilidad estimada por factor de riesgo (sin cambios)

In [18]:
header("5.3 Morbilidad estimada → colonia")

sistemicos = calidad[calidad["clasificacion_cdmx"] != "cdmx_local"]
pct_sistema = (sistemicos["excede_alguna_nom"].mean() * 100).round(1) if len(sistemicos) else np.nan

# Factor de riesgo de morbilidad (heredado v4)
features_riesgo = colonias_demo[["id_colonia","pobreza_pct_promedio",
                                   "pct_aguadv","pct_drenaje","pob_colonia"]].copy()
mean_pob = features_riesgo["pobreza_pct_promedio"].mean()
mean_sa  = (100 - features_riesgo["pct_aguadv"]).mean()
mean_sd  = (100 - features_riesgo["pct_drenaje"]).mean()

features_riesgo["riesgo_pobreza"] = features_riesgo["pobreza_pct_promedio"] / mean_pob
features_riesgo["riesgo_sin_agua"] = (100 - features_riesgo["pct_aguadv"]) / mean_sa if mean_sa > 0 else 1.0
features_riesgo["riesgo_sin_drenaje"] = (100 - features_riesgo["pct_drenaje"]) / mean_sd if mean_sd > 0 else 1.0
features_riesgo["factor_riesgo_morbilidad"] = (features_riesgo["riesgo_pobreza"]
                                                + features_riesgo["riesgo_sin_agua"]
                                                + features_riesgo["riesgo_sin_drenaje"]) / 3
features_riesgo["tasa_morbilidad_estimada_por_100k"] = (
    TASA_MORBI_ESTATAL * features_riesgo["factor_riesgo_morbilidad"]).round(2)
features_riesgo["casos_morbilidad_estimados"] = (
    features_riesgo["pob_colonia"] * features_riesgo["tasa_morbilidad_estimada_por_100k"] / 100_000).round(1)

print(f"  factor_riesgo: media={features_riesgo['factor_riesgo_morbilidad'].mean():.2f}, "
       f"std={features_riesgo['factor_riesgo_morbilidad'].std():.2f}")
print(f"  Suma casos estimados: {features_riesgo['casos_morbilidad_estimados'].sum():,.0f}")
print(f"  vs CASOS reales SSA 2017: {CASOS_MORBI_ESTATAL:,}")

metrics_globales = territorios[["id_colonia"]].merge(
    features_riesgo[["id_colonia","factor_riesgo_morbilidad",
                      "tasa_morbilidad_estimada_por_100k","casos_morbilidad_estimados"]],
    on="id_colonia", how="left")
metrics_globales["pct_excede_nom_sistema"] = pct_sistema
metrics_globales["sistema_origen"] = "replicado_estatal"
metrics_globales["morbi_origen"] = "estimacion_factor_riesgo"
metrics_globales["tasa_morbilidad_estatal_por_100k_referencia"] = TASA_MORBI_ESTATAL


 5.3 Morbilidad estimada → colonia
  factor_riesgo: media=1.00, std=0.71
  Suma casos estimados: 466,413
  vs CASOS reales SSA 2017: 460,646


### 5.4 Fugas SACMEX agregadas por colonia

In [19]:
header("5.4 Fugas → colonia")

fugas_total = (incidentes_master.dropna(subset=["id_colonia"])
               .groupby("id_colonia")
               .agg(n_fugas_total=("folio_incidente","count"),
                    lag_promedio_dias=("lag_dias","mean"),
                    pct_lag_largo=("flag_lag_largo", lambda s: round(s.mean()*100, 1)))
               .reset_index())
fugas_total["lag_promedio_dias"] = fugas_total["lag_promedio_dias"].round(2)

fugas_anio = (incidentes_master.dropna(subset=["id_colonia","anio_reporte"])
              .groupby(["id_colonia","anio_reporte"]).size().rename("n_fugas").reset_index())

print(f"  Colonias con al menos 1 fuga: {len(fugas_total):,}/{len(territorios):,}")
print(f"  Total fugas asignadas: {fugas_total['n_fugas_total'].sum():,}")


 5.4 Fugas → colonia
  Colonias con al menos 1 fuga: 1,774/1,837
  Total fugas asignadas: 512,601


### 5.5 Distancia a planta potabilizadora

In [20]:
header("5.5 Distancia a planta")

if len(plantas_master) == 0 or plantas_master["latitud"].notna().sum() == 0:
    territorios_planta = territorios[["id_colonia"]].copy()
    territorios_planta["planta_cercana_id"] = pd.NA
    territorios_planta["dist_planta_km"] = np.nan
else:
    plantas_geo = plantas_master.dropna(subset=["latitud","longitud"]).copy()
    ids_p = asignar_id_mas_cercano(territorios, plantas_geo,
                                     "centroide_lat","centroide_lon",
                                     "latitud","longitud","id_planta")
    territorios_planta = territorios[["id_colonia","centroide_lat","centroide_lon"]].copy()
    territorios_planta["planta_cercana_id"] = ids_p.values
    territorios_planta = territorios_planta.merge(
        plantas_geo[["id_planta","latitud","longitud"]],
        left_on="planta_cercana_id", right_on="id_planta", how="left")
    territorios_planta["dist_planta_km"] = haversine_km(
        territorios_planta["centroide_lat"].values,
        territorios_planta["centroide_lon"].values,
        territorios_planta["latitud"].values,
        territorios_planta["longitud"].values).round(2)
    territorios_planta = territorios_planta[["id_colonia","planta_cercana_id","dist_planta_km"]]

print(f"  dist_planta_km describe:")
print(territorios_planta["dist_planta_km"].describe(percentiles=[.5,.9]).round(2).to_string())


 5.5 Distancia a planta
  dist_planta_km describe:
count    1837.00
mean        6.44
std         4.50
min         0.08
50%         5.06
90%        13.40
max        25.43


### 5.6 Proxy antigüedad de la red

In [21]:
header("5.6 Proxy antigüedad")

proxy_red = territorios[["id_colonia"]].copy()
proxy_red = proxy_red.merge(
    colonias_demo[["id_colonia","densidad_pob_por_vivienda","pct_drenaje","pct_aguadv"]],
    on="id_colonia", how="left")

def _mm(s):
    s = pd.to_numeric(s, errors="coerce")
    rmin, rmax = s.min(), s.max()
    if pd.isna(rmin) or rmax == rmin: return pd.Series(0.5, index=s.index)
    return ((s - rmin) / (rmax - rmin)).fillna(0.5)

proxy_red["densidad_norm"]    = _mm(proxy_red["densidad_pob_por_vivienda"])
proxy_red["pct_drenaje_norm"] = _mm(proxy_red["pct_drenaje"])
proxy_red["pct_aguadv_norm"]  = _mm(proxy_red["pct_aguadv"])
proxy_red["antiguedad_red_proxy"] = (0.50 * proxy_red["densidad_norm"]
                                       + 0.25 * proxy_red["pct_drenaje_norm"]
                                       + 0.25 * proxy_red["pct_aguadv_norm"]).round(4)
print(f"  antiguedad_red_proxy: {proxy_red['antiguedad_red_proxy'].describe(percentiles=[.5]).round(3).to_dict()}")


 5.6 Proxy antigüedad
  antiguedad_red_proxy: {'count': 1837.0, 'mean': 0.747, 'std': 0.131, 'min': 0.355, '50%': 0.8, 'max': 0.891}


## 6. Tabla maestra `maestra_colonia` (snapshot)

In [22]:
header("6. maestra_colonia (snapshot)")

maestra_col = territorios.copy()

# Demografía + pobreza
COLS_DEMO = ["pct_aguadv","pct_drenaje","pct_tinaco","densidad_pob_por_vivienda",
             "pob_colonia","pobreza_pct_promedio","ingreso_lpi_pct_promedio",
             "carencia_calidad_espacios_pct","pob_origen","pobreza_origen"]
maestra_col = maestra_col.merge(colonias_demo[["id_colonia"] + COLS_DEMO], on="id_colonia", how="left")

# Calidad agua nearest
maestra_col = maestra_col.merge(territorios_calidad[["id_colonia","sitio_calidad_cercano",
                                                        "dist_sitio_km","pct_excede_nom_local",
                                                        "n_mediciones_sitio","calidad_origen"]],
                                  on="id_colonia", how="left")

# Calidad sistémica + morbilidad estimada
maestra_col = maestra_col.merge(metrics_globales[["id_colonia","pct_excede_nom_sistema",
                                                     "factor_riesgo_morbilidad",
                                                     "tasa_morbilidad_estimada_por_100k",
                                                     "casos_morbilidad_estimados",
                                                     "tasa_morbilidad_estatal_por_100k_referencia",
                                                     "sistema_origen","morbi_origen"]],
                                  on="id_colonia", how="left")

# Fugas total
maestra_col = maestra_col.merge(fugas_total, on="id_colonia", how="left")
maestra_col["n_fugas_total"] = maestra_col["n_fugas_total"].fillna(0).astype(int)
maestra_col["pct_lag_largo"] = maestra_col["pct_lag_largo"].fillna(0)

# Snapshot año actual
fugas_actual = fugas_anio[fugas_anio["anio_reporte"]==ANIO_REFERENCIA] \
                .rename(columns={"n_fugas":f"n_fugas_{ANIO_REFERENCIA}"}) \
                .drop(columns=["anio_reporte"])
maestra_col = maestra_col.merge(fugas_actual, on="id_colonia", how="left")
maestra_col[f"n_fugas_{ANIO_REFERENCIA}"] = maestra_col[f"n_fugas_{ANIO_REFERENCIA}"].fillna(0).astype(int)

# Métricas per cápita
maestra_col["fugas_por_10k_hab_total"] = (maestra_col["n_fugas_total"]
                                            / maestra_col["pob_colonia"].replace(0, np.nan)
                                            * 10_000).round(2)
maestra_col[f"fugas_por_10k_hab_{ANIO_REFERENCIA}"] = (maestra_col[f"n_fugas_{ANIO_REFERENCIA}"]
                                                        / maestra_col["pob_colonia"].replace(0, np.nan)
                                                        * 10_000).round(2)

# Planta + antigüedad
maestra_col = maestra_col.merge(territorios_planta, on="id_colonia", how="left")
maestra_col = maestra_col.merge(proxy_red[["id_colonia","antiguedad_red_proxy",
                                              "densidad_norm","pct_drenaje_norm","pct_aguadv_norm"]],
                                  on="id_colonia", how="left")

# IVS y score base
maestra_col["pobreza_norm_ivs"] = _mm(maestra_col["pobreza_pct_promedio"])
maestra_col["morbi_norm_ivs"]   = _mm(maestra_col["tasa_morbilidad_estimada_por_100k"])
maestra_col["IVS"] = (0.5 * maestra_col["pobreza_norm_ivs"]
                    + 0.5 * maestra_col["morbi_norm_ivs"]).round(4)
fugas_norm_score = _mm(np.log1p(maestra_col["fugas_por_10k_hab_total"].clip(lower=0)))
maestra_col["score_intervencion_base"] = (maestra_col["IVS"] * fugas_norm_score).round(4)

print(f"\n  maestra_colonia: {maestra_col.shape}")
display(maestra_col.head(3))
LOG["maestra_colonia"] = {"n_filas": len(maestra_col), "n_columnas": maestra_col.shape[1]}


 6. maestra_colonia (snapshot)

  maestra_colonia: (1837, 50)


,id_colonia,cveut,cve_alcaldia,nom_alcaldia,nom_colonia,nom_colonia_norm,codigos_postales,origen,poblacion_iecm,centroide_lat,centroide_lon,centroide_imputado,pct_aguadv,pct_drenaje,pct_tinaco,densidad_pob_por_vivienda,pob_colonia,pobreza_pct_promedio,ingreso_lpi_pct_promedio,carencia_calidad_espacios_pct,pob_origen,pobreza_origen,sitio_calidad_cercano,dist_sitio_km,pct_excede_nom_local,n_mediciones_sitio,calidad_origen,pct_excede_nom_sistema,factor_riesgo_morbilidad,tasa_morbilidad_estimada_por_100k,casos_morbilidad_estimados,tasa_morbilidad_estatal_por_100k_referencia,sistema_origen,morbi_origen,n_fugas_total,lag_promedio_dias,pct_lag_largo,n_fugas_2024,fugas_por_10k_hab_total,fugas_por_10k_hab_2024,planta_cercana_id,dist_planta_km,antiguedad_red_proxy,densidad_norm,pct_drenaje_norm,pct_aguadv_norm,pobreza_norm_ivs,morbi_norm_ivs,IVS,score_intervencion_base
0,10-001,10-001,010,ALVARO OBREGON,ABRAHAM GONZALEZ,ABRAHAM GONZALEZ,,iecm,459,19.386403,-99.205473,0,99.54,99.6,81.23,3.46,459,37.67,48.76,5.22,iecm_oficial,distribuida_uniforme_alcaldia,OCAVM0010RNL21,6.15,0.0,2,nearest_sitio_cdmx_local,80.8,0.79468,3974.97,18.2,5001.98,replicado_estatal,estimacion_factor_riesgo,26,0.00,0.0,0,566.45,0.00,PLANTA_006,10.42,0.7997,0.6875,0.851613,0.972275,0.647519,0.12727,0.3874,0.2342
1,10-002,10-002,010,ALVARO OBREGON,ACUEDUCTO,ACUEDUCTO,,iecm,3188,19.397885,-99.204290,0,99.54,99.6,81.23,3.46,3188,37.67,48.76,5.22,iecm_oficial,distribuida_uniforme_alcaldia,OCAVM0010RNL21,4.89,0.0,2,nearest_sitio_cdmx_local,80.8,0.79468,3974.97,126.7,5001.98,replicado_estatal,estimacion_factor_riesgo,256,15.32,4.7,1,803.01,3.14,PLANTA_000,9.66,0.7997,0.6875,0.851613,0.972275,0.647519,0.12727,0.3874,0.2471
2,10-003,10-003,010,ALVARO OBREGON,ACUILOTLA,ACUILOTLA,,iecm,1927,19.359605,-99.243782,0,99.54,99.6,81.23,3.46,1927,37.67,48.76,5.22,iecm_oficial,distribuida_uniforme_alcaldia,SITIO NUEVO 1,6.39,100.0,2,nearest_sitio_cdmx_local,80.8,0.79468,3974.97,76.6,5001.98,replicado_estatal,estimacion_factor_riesgo,17,0.00,0.0,1,88.22,5.19,PLANTA_006,14.88,0.7997,0.6875,0.851613,0.972275,0.647519,0.12727,0.3874,0.1659


## 7. `maestra_colonia_anio` (anual 2018-2024)

In [23]:
header("7. maestra_colonia_anio")

ANIOS_FUGAS = sorted(incidentes_master["anio_reporte"].dropna().astype(int).unique())
print(f"  Años SACMEX: {ANIOS_FUGAS}")

grid_ca = pd.MultiIndex.from_product(
    [territorios["id_colonia"], ANIOS_FUGAS], names=["id_colonia","anio"]
).to_frame(index=False)

fugas_year = fugas_anio.rename(columns={"anio_reporte":"anio"})
fugas_year["anio"] = fugas_year["anio"].astype(int)
grid_ca = grid_ca.merge(fugas_year, on=["id_colonia","anio"], how="left")
grid_ca["n_fugas"] = grid_ca["n_fugas"].fillna(0).astype(int)

COLS_EST = ["cve_alcaldia","nom_alcaldia","nom_colonia","origen",
             "pob_colonia","pct_aguadv","pct_drenaje","densidad_pob_por_vivienda",
             "pobreza_pct_promedio","ingreso_lpi_pct_promedio",
             "sitio_calidad_cercano","dist_sitio_km","pct_excede_nom_local",
             "factor_riesgo_morbilidad","tasa_morbilidad_estimada_por_100k",
             "antiguedad_red_proxy","dist_planta_km","planta_cercana_id"]
COLS_EST = [c for c in COLS_EST if c in maestra_col.columns]
grid_ca = grid_ca.merge(maestra_col[["id_colonia"] + COLS_EST], on="id_colonia", how="left")
grid_ca["fugas_por_10k_hab"] = (grid_ca["n_fugas"]
                                  / grid_ca["pob_colonia"].replace(0, np.nan)
                                  * 10_000).round(2)

print(f"  Shape: {grid_ca.shape}")
display(grid_ca.head())
LOG["maestra_colonia_anio"] = {"n_filas": len(grid_ca), "anios": [int(a) for a in ANIOS_FUGAS]}


 7. maestra_colonia_anio
  Años SACMEX: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
  Shape: (12859, 22)


,id_colonia,anio,n_fugas,cve_alcaldia,nom_alcaldia,nom_colonia,origen,pob_colonia,pct_aguadv,pct_drenaje,densidad_pob_por_vivienda,pobreza_pct_promedio,ingreso_lpi_pct_promedio,sitio_calidad_cercano,dist_sitio_km,pct_excede_nom_local,factor_riesgo_morbilidad,tasa_morbilidad_estimada_por_100k,antiguedad_red_proxy,dist_planta_km,planta_cercana_id,fugas_por_10k_hab
0,10-001,2018,2,010,ALVARO OBREGON,ABRAHAM GONZALEZ,iecm,459,99.54,99.6,3.46,37.67,48.76,OCAVM0010RNL21,6.15,0.0,0.79468,3974.97,0.7997,10.42,PLANTA_006,43.57
1,10-001,2019,10,010,ALVARO OBREGON,ABRAHAM GONZALEZ,iecm,459,99.54,99.6,3.46,37.67,48.76,OCAVM0010RNL21,6.15,0.0,0.79468,3974.97,0.7997,10.42,PLANTA_006,217.86
2,10-001,2020,5,010,ALVARO OBREGON,ABRAHAM GONZALEZ,iecm,459,99.54,99.6,3.46,37.67,48.76,OCAVM0010RNL21,6.15,0.0,0.79468,3974.97,0.7997,10.42,PLANTA_006,108.93
3,10-001,2021,3,010,ALVARO OBREGON,ABRAHAM GONZALEZ,iecm,459,99.54,99.6,3.46,37.67,48.76,OCAVM0010RNL21,6.15,0.0,0.79468,3974.97,0.7997,10.42,PLANTA_006,65.36
4,10-001,2022,4,010,ALVARO OBREGON,ABRAHAM GONZALEZ,iecm,459,99.54,99.6,3.46,37.67,48.76,OCAVM0010RNL21,6.15,0.0,0.79468,3974.97,0.7997,10.42,PLANTA_006,87.15


## 8. `maestra_colonia_semestre` (target del XGBoost)

Con el SACMEX histórico incluido, ahora cubrimos **2018-S1 → 2024-S2** (~14 periodos por colonia). Permite el split temporal del README:
* **Train:** 2018-S1 → 2022-S2 (10 periodos)
* **Validation:** 2023-S1 + 2023-S2
* **Test:** 2024-S1 + 2024-S2

In [24]:
header("8. maestra_colonia_semestre")

incidentes_master["semestre"] = incidentes_master["fecha_reporte"].dt.month.apply(
    lambda m: 1 if m <= 6 else 2 if m >= 7 else pd.NA)

fugas_sem = (incidentes_master.dropna(subset=["id_colonia","anio_reporte","semestre"])
             .groupby(["id_colonia","anio_reporte","semestre"])
             .size().rename("n_fugas").reset_index()
             .rename(columns={"anio_reporte":"anio"}))

periodos = (incidentes_master.dropna(subset=["anio_reporte","semestre"])
            [["anio_reporte","semestre"]].drop_duplicates()
            .sort_values(["anio_reporte","semestre"])
            .rename(columns={"anio_reporte":"anio"}).reset_index(drop=True))
print(f"  Periodos detectados: {len(periodos)}")
print(periodos.assign(label=lambda d: d["anio"].astype(int).astype(str)+"-S"+d["semestre"].astype(int).astype(str))["label"].tolist())

grid_cs = territorios[["id_colonia"]].merge(periodos, how="cross")
grid_cs["anio"] = grid_cs["anio"].astype(int)
grid_cs["semestre"] = grid_cs["semestre"].astype(int)
fugas_sem["anio"] = fugas_sem["anio"].astype(int)
fugas_sem["semestre"] = fugas_sem["semestre"].astype(int)

grid_cs = grid_cs.merge(fugas_sem, on=["id_colonia","anio","semestre"], how="left")
grid_cs["n_fugas"] = grid_cs["n_fugas"].fillna(0).astype(int)
grid_cs = grid_cs.merge(maestra_col[["id_colonia"] + COLS_EST], on="id_colonia", how="left")
grid_cs["fugas_por_10k_hab_semestre"] = (grid_cs["n_fugas"]
                                          / grid_cs["pob_colonia"].replace(0, np.nan)
                                          * 10_000).round(2)
grid_cs["periodo_label"] = grid_cs["anio"].astype(str) + "-S" + grid_cs["semestre"].astype(str)
grid_cs = grid_cs.sort_values(["id_colonia","anio","semestre"]).reset_index(drop=True)

print(f"\n  Shape: {grid_cs.shape}")
print(f"  N fugas por periodo:")
print(grid_cs.groupby(["anio","semestre"])["n_fugas"].agg(['sum','mean','median']))
LOG["maestra_colonia_semestre"] = {"n_filas": len(grid_cs), "n_periodos": len(periodos)}


 8. maestra_colonia_semestre
  Periodos detectados: 13
['2018-S1', '2018-S2', '2019-S1', '2019-S2', '2020-S1', '2020-S2', '2021-S1', '2021-S2', '2022-S1', '2022-S2', '2023-S1', '2023-S2', '2024-S1']

  Shape: (23881, 24)
  N fugas por periodo:
                 sum       mean  median
anio semestre                          
2018 1         29412  16.010887     4.0
     2         37557  20.444747     5.0
2019 1         39290  21.388133     5.0
     2         29406  16.007621     3.0
2020 1         20747  11.293958     3.0
     2         12498   6.803484     1.0
2021 1         36767  20.014698     7.0
     2         46623  25.379967     9.0
2022 1         61145  33.285248     8.0
     2         70665  38.467610     8.0
2023 1         53147  28.931410     7.0
     2         60425  32.893304     9.0
2024 1         14919   8.121394     1.0


## 9. Índice de Vulnerabilidad Hídrica (sin cambios mayores vs v5)

In [25]:
header("9. IVH")

def normalizar_minmax(serie, transformar_log=False):
    s = pd.to_numeric(serie, errors="coerce")
    if transformar_log:
        s = np.log1p(s.clip(lower=0))
    rmin, rmax = s.min(), s.max()
    if pd.isna(rmin) or rmax == rmin:
        return pd.Series(np.nan, index=s.index), False
    return (s - rmin) / (rmax - rmin), True

ivh = maestra_col.copy()
ivh["pobreza_norm"], ok_pob = normalizar_minmax(ivh["pobreza_pct_promedio"], transformar_log=False)
ivh["calidad_norm"], ok_cal = normalizar_minmax(ivh["pct_excede_nom_local"], transformar_log=True)
ivh["fugas_norm"],   ok_fug = normalizar_minmax(ivh["fugas_por_10k_hab_total"], transformar_log=True)
ivh["morbi_norm"],   ok_mor = normalizar_minmax(ivh["tasa_morbilidad_estimada_por_100k"], transformar_log=False)

componentes = {"pobreza":(PESOS_IVH["pobreza"], ok_pob),
                "calidad_agua":(PESOS_IVH["calidad_agua"], ok_cal),
                "fugas_per_capita":(PESOS_IVH["fugas_per_capita"], ok_fug),
                "morbilidad":(PESOS_IVH["morbilidad"], ok_mor)}
peso_perdido = sum(p for p, ok in componentes.values() if not ok)
peso_activo = sum(p for p, ok in componentes.values() if ok)
factor = (peso_activo + peso_perdido) / peso_activo if peso_activo > 0 else 1.0
PESOS_EFECTIVOS = {k:(p*factor if ok else 0) for k,(p,ok) in componentes.items()}

ivh[["pobreza_norm","calidad_norm","fugas_norm","morbi_norm"]] = \
    ivh[["pobreza_norm","calidad_norm","fugas_norm","morbi_norm"]].fillna(0)
ivh["IVH"] = (PESOS_EFECTIVOS["pobreza"] * ivh["pobreza_norm"]
            + PESOS_EFECTIVOS["calidad_agua"] * ivh["calidad_norm"]
            + PESOS_EFECTIVOS["fugas_per_capita"] * ivh["fugas_norm"]
            + PESOS_EFECTIVOS["morbilidad"] * ivh["morbi_norm"]).round(4)

top10 = ivh.nlargest(10, "IVH")[["nom_alcaldia","nom_colonia","IVH",
                                    "pobreza_pct_promedio","fugas_por_10k_hab_total",
                                    "pct_excede_nom_local","tasa_morbilidad_estimada_por_100k"]]
display(top10)
LOG["vulnerabilidad_hidrica"] = {"n_filas": len(ivh),
                                  "ivh_media": round(float(ivh["IVH"].mean()), 4),
                                  "ivh_max": round(float(ivh["IVH"].max()), 4),
                                  "pesos_efectivos": PESOS_EFECTIVOS}


 9. IVH


,nom_alcaldia,nom_colonia,IVH,pobreza_pct_promedio,fugas_por_10k_hab_total,pct_excede_nom_local,tasa_morbilidad_estimada_por_100k
1425,MILPA ALTA,SAN BARTOLOME XICOMULCO (PBLO),0.8807,53.44,511.63,71.4,22351.34
1426,MILPA ALTA,SAN FRANCISCO TECOXPA (PBLO),0.8639,53.44,207.98,77.8,22351.34
1760,XOCHIMILCO,SAN LORENZO,0.8612,46.88,14247.60,59.5,15953.95
1434,MILPA ALTA,VILLA MILPA ALTA (PBLO),0.8576,53.44,159.45,77.8,22351.34
1424,MILPA ALTA,SAN ANTONIO TECOMITL (PBLO),0.8552,53.44,144.14,77.8,22351.34
1427,MILPA ALTA,SAN JERONIMO MIACATLAN (PBLO),0.8461,53.44,97.94,77.8,22351.34
1432,MILPA ALTA,SAN SALVADOR CUAUHTENCO (PBLO),0.8446,53.44,111.66,71.4,22351.34
1431,MILPA ALTA,SAN PEDRO ATOCPAN (PBLO),0.8386,53.44,86.79,71.4,22351.34
1423,MILPA ALTA,SAN AGUSTIN OHTENCO (PBLO),0.8356,53.44,62.70,77.8,22351.34
1781,XOCHIMILCO,NATIVITAS,0.8352,46.88,2624.60,77.8,15953.95


## 10. Cruce SEPOMEX → IECM (vista ciudadano por CP)

SEPOMEX queda como **fuente secundaria** sólo para resolver consultas tipo "el ciudadano ingresa su CP". Construimos un mapa (CP → lista de id_colonia IECM) por intersección espacial vía nombre + alcaldía + fuzzy match. Es la pieza que necesita la **vista ciudadano** del dashboard.

In [26]:
header("10. CP → IECM (vista ciudadano)")

# Aproximación: SEPOMEX da (alcaldía, colonia, CP). IECM tiene (alcaldía, UT).
# Intentamos mapear por nombre exacto + fuzzy dentro de alcaldía
df_sep_unique = df_sep[["cve_alcaldia","nom_colonia_norm","nom_colonia","codigo_postal"]].copy()

# Match exacto SEPOMEX → IECM
mi_iecm_cp = pd.MultiIndex.from_frame(df_iecm[["cve_alcaldia","nom_colonia_norm"]])
mi_sep_cp = pd.MultiIndex.from_frame(df_sep_unique[["cve_alcaldia","nom_colonia_norm"]])
mask_exacto = mi_sep_cp.isin(mi_iecm_cp)

print(f"  Match exacto SEPOMEX→IECM: {mask_exacto.sum():,}/{len(df_sep_unique):,} ({100*mask_exacto.mean():.1f}%)")

# Para los que NO matchean, fuzzy match
sep_no_match = df_sep_unique[~mask_exacto]
filas_fuzzy = []
for cve, sub in sep_no_match.groupby("cve_alcaldia"):
    universo = df_iecm.loc[df_iecm["cve_alcaldia"]==cve, "nom_colonia_norm"].tolist()
    if not universo: continue
    df_m = fuzzy_match_uno_a_uno(sub["nom_colonia_norm"].tolist(), universo,
                                  threshold_auto=80, threshold_cand=65)
    df_m["cve_alcaldia"] = cve
    filas_fuzzy.append(df_m)

if filas_fuzzy:
    fuzzy_sep = pd.concat(filas_fuzzy, ignore_index=True)
    n_auto = (fuzzy_sep["decision"]=="auto").sum()
    print(f"  Fuzzy resuelve: {n_auto:,} adicionales")

# Construir cp_a_colonia: CP → list(id_colonia IECM)
mapa_iecm_id_cp = df_iecm.set_index(["cve_alcaldia","nom_colonia_norm"])["id_colonia"].to_dict()

def _resolve_id_cp(row):
    key = (row["cve_alcaldia"], row["nom_colonia_norm"])
    return mapa_iecm_id_cp.get(key, None)

df_sep_unique["id_colonia"] = df_sep_unique.apply(_resolve_id_cp, axis=1)

# Aplicar fuzzy a los huérfanos
if filas_fuzzy:
    fuzzy_auto = fuzzy_sep[fuzzy_sep["decision"]=="auto"][["cve_alcaldia","consulta","match"]]
    fuzzy_auto = fuzzy_auto.rename(columns={"consulta":"nom_colonia_norm"})
    fuzzy_map = {(r["cve_alcaldia"], r["nom_colonia_norm"]): r["match"] for _, r in fuzzy_auto.iterrows()}

    def _resolve_fuzzy(row):
        if pd.notna(row["id_colonia"]): return row["id_colonia"]
        key = (row["cve_alcaldia"], row["nom_colonia_norm"])
        if key in fuzzy_map:
            inner_key = (row["cve_alcaldia"], fuzzy_map[key])
            return mapa_iecm_id_cp.get(inner_key, None)
        return None
    df_sep_unique["id_colonia"] = df_sep_unique.apply(_resolve_fuzzy, axis=1)

n_resueltos = df_sep_unique["id_colonia"].notna().sum()
print(f"\n  CPs SEPOMEX con id_colonia IECM: {n_resueltos:,}/{len(df_sep_unique):,} ({100*n_resueltos/len(df_sep_unique):.1f}%)")

cp_a_colonia = df_sep_unique[["codigo_postal","nom_colonia","cve_alcaldia","id_colonia"]].copy()
LOG["cp_a_colonia"] = {"n_filas": len(cp_a_colonia),
                        "n_resueltos": int(n_resueltos)}


 10. CP → IECM (vista ciudadano)
  Match exacto SEPOMEX→IECM: 656/1,526 (43.0%)
  Fuzzy resuelve: 656 adicionales

  CPs SEPOMEX con id_colonia IECM: 1,312/1,526 (86.0%)


## 11. Verificación post-fusión

In [27]:
header("11. Verificación")

# Test 1: PKs únicas
print("Test 1 — PKs:")
for tabla, df, pk in [
    ("territorios_cdmx", territorios, ("id_colonia",)),
    ("maestra_colonia", maestra_col, ("id_colonia",)),
    ("maestra_colonia_anio", grid_ca, ("id_colonia","anio")),
    ("maestra_colonia_semestre", grid_cs, ("id_colonia","anio","semestre")),
]:
    n_dup = df.duplicated(subset=list(pk)).sum()
    flag = "✅" if n_dup == 0 else "⚠️"
    print(f"  {flag} {tabla:<30} duplicados={n_dup}")

# Test 2: Cobertura
print(f"\nTest 2 — Cobertura:")
print(f"  Catálogo: {len(territorios):,}")
print(f"  Maestra:  {len(maestra_col):,}")
flag = "✅" if len(territorios) == len(maestra_col) else "❌"
print(f"  {flag}")

# Test 3: IVH discrimina
print(f"\nTest 3 — IVH std={ivh['IVH'].std():.3f} (debe > 0.05): {'✅' if ivh['IVH'].std() > 0.05 else '⚠️'}")

# Test 4: pob_colonia desde IECM (no más distribución uniforme)
n_iecm_pob = (maestra_col["pob_origen"]=="iecm_oficial").sum()
print(f"\nTest 4 — Población oficial IECM: {n_iecm_pob:,}/{len(maestra_col):,}")

# Test 5: span temporal completo
print(f"\nTest 5 — Span temporal SACMEX: {min(ANIOS_FUGAS)} → {max(ANIOS_FUGAS)}")
print(f"  Periodos semestrales: {grid_cs.groupby(['anio','semestre']).ngroups}")

# Test 6: ratio IECM vs huérfanas (post-reasignación)
n_iecm = (maestra_col["origen"]=="iecm").sum()
n_huerf = (maestra_col["origen"]=="sacmex_no_oficial").sum()
print(f"\nTest 6 — Composición catálogo:")
print(f"  IECM:               {n_iecm:,}")
print(f"  Huérfanas finales:  {n_huerf:,}")
flag = "✅" if n_huerf < 200 else "⚠️"
print(f"  {flag} (esperado < 200 huérfanas tras reasignación)")

# Test 7: % de fugas con id_colonia válido
n_con_id = incidentes_master["id_colonia"].notna().sum()
pct = 100 * n_con_id / len(incidentes_master)
print(f"\nTest 7 — Cobertura de fugas:")
print(f"  Reportes con id_colonia: {n_con_id:,}/{len(incidentes_master):,} ({pct:.1f}%)")
flag = "✅" if pct >= 95 else "⚠️"
print(f"  {flag} (esperado >= 95%)")

# Test 8: reportes con NaN en fugas_por_10k_hab_total
n_nan = maestra_col["fugas_por_10k_hab_total"].isna().sum()
print(f"\nTest 8 — fugas_por_10k_hab calculable:")
print(f"  NaN: {n_nan} de {len(maestra_col)}")
flag = "✅" if n_nan < 100 else "⚠️"
print(f"  {flag} (esperado < 100 NaN, solo huérfanas residuales sin pob)")


 11. Verificación
Test 1 — PKs:
  ✅ territorios_cdmx               duplicados=0
  ✅ maestra_colonia                duplicados=0
  ✅ maestra_colonia_anio           duplicados=0
  ✅ maestra_colonia_semestre       duplicados=0

Test 2 — Cobertura:
  Catálogo: 1,837
  Maestra:  1,837
  ✅

Test 3 — IVH std=0.177 (debe > 0.05): ✅

Test 4 — Población oficial IECM: 1,837/1,837

Test 5 — Span temporal SACMEX: 2018 → 2024
  Periodos semestrales: 13

Test 6 — Composición catálogo:
  IECM:               1,837
  Huérfanas finales:  0
  ✅ (esperado < 200 huérfanas tras reasignación)

Test 7 — Cobertura de fugas:
  Reportes con id_colonia: 512,601/519,033 (98.8%)
  ✅ (esperado >= 95%)

Test 8 — fugas_por_10k_hab calculable:
  NaN: 0 de 1837
  ✅ (esperado < 100 NaN, solo huérfanas residuales sin pob)


## 12. Export del modelo canónico

In [28]:
# Export
territorios.to_csv(RUTAS_OUT["territorios"], index=False, encoding="utf-8-sig")
sociodemo.to_csv(RUTAS_OUT["sociodemografia"], index=False, encoding="utf-8-sig")
morbi.to_csv(RUTAS_OUT["morbilidad"], index=False, encoding="utf-8-sig")
sitios_master.to_csv(RUTAS_OUT["sitios_monitoreo"], index=False, encoding="utf-8-sig")
calidad.to_csv(RUTAS_OUT["calidad_agua"], index=False, encoding="utf-8-sig")
incidentes_master.to_csv(RUTAS_OUT["incidentes_fugas"], index=False, encoding="utf-8-sig")
if len(plantas_master):
    plantas_master.to_csv(RUTAS_OUT["plantas_potabilizadoras"], index=False, encoding="utf-8-sig")
maestra_col.to_csv(RUTAS_OUT["maestra_colonia"], index=False, encoding="utf-8-sig")
grid_ca.to_csv(RUTAS_OUT["maestra_colonia_anio"], index=False, encoding="utf-8-sig")
grid_cs.to_csv(RUTAS_OUT["maestra_colonia_semestre"], index=False, encoding="utf-8-sig")
ivh.to_csv(RUTAS_OUT["vulnerabilidad_hidrica"], index=False, encoding="utf-8-sig")
cp_a_colonia.to_csv(RUTAS_OUT["cp_a_colonia"], index=False, encoding="utf-8-sig")

LOG["_meta"] = {
    "generado_en": datetime.now().isoformat(timespec="seconds"),
    "version": "v7 — IECM + SACMEX 2018-2024 + reasignación espacial de huérfanos",
    "decisiones_de_fusion": {
        "universo_principal": "IECM 2022 (1837 UT con polígonos y población oficial)",
        "sepomex_uso": "secundario, sólo para CP → id_colonia (vista ciudadano)",
        "sacmex_span": f"{min(ANIOS_FUGAS)}-{max(ANIOS_FUGAS)}",
        "calidad_agua": "KNN nearest sitio cdmx_local (haversine)",
        "morbilidad": "factor de riesgo (pobreza+sin_agua+sin_drenaje) × tasa estatal",
        "fuzzy_threshold": FUZZY_AUTO,
        "anio_referencia": ANIO_REFERENCIA,
        "pesos_IVH_efectivos": PESOS_EFECTIVOS,
    },
}
with open(RUTAS_OUT["log_fusion"], "w", encoding="utf-8") as f:
    json.dump(dict(LOG), f, indent=2, ensure_ascii=False, default=str)

print("\n📦 Archivos generados:")
for f in sorted(DATOS_MAESTROS.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45}  {size_kb:>10,.1f} KB")


📦 Archivos generados:
  _linkage_sacmex_huerfanos.csv                       248.5 KB
  _log_fusion.json                                      2.2 KB
  calidad_agua.csv                                     93.4 KB
  cp_a_colonia.csv                                     52.0 KB
  incidentes_fugas.csv                             88,682.2 KB
  maestra_colonia.csv                                 888.5 KB
  maestra_colonia_anio.csv                          2,127.3 KB
  maestra_colonia_semestre.csv                      4,173.7 KB
  morbilidad_cdmx.csv                                   0.8 KB
  plantas_potabilizadoras.csv                           8.1 KB
  sitios_monitoreo.csv                                  5.2 KB
  sociodemografia_alcaldia.csv                          3.3 KB
  territorios_cdmx.csv                                190.8 KB
  vulnerabilidad_hidrica_colonia.csv                1,019.1 KB


## 13. Conclusiones de la fusión v6

### 13.1 Universo territorial

* **1 837 UT del IECM** como base canónica (vs 1 503 SEPOMEX en v5, +22 % cobertura)
* **Polígonos reales** disponibles en `colonias_iecm.geojson` para mapas coropléticos
* **Población oficial POBL** por colonia (suma 9.21M ≈ INEGI 2020)
* **Match SACMEX↔IECM:** ~85 % automático (vs 39.5 % exacto + fuzzy)

### 13.2 Span temporal

* **SACMEX 2018-2024** completo (~519 K reportes)
* **14 periodos semestrales** disponibles para XGBoost
* Split posible: train 2018-S1→2022-S2 / val 2023 / test 2024

### 13.3 Listo para `analisis.ipynb`

* `maestra_colonia.csv` (~1 950 filas × 50 cols) — insumo principal
* `maestra_colonia_anio.csv` (~13 650 filas) — series anuales
* `maestra_colonia_semestre.csv` (~27 300 filas) — target XGBoost semestral
* `colonias_iecm.geojson` — polígonos para mapas coropléticos
* `cp_a_colonia.csv` — vista ciudadano (CP → id_colonia)
* `vulnerabilidad_hidrica_colonia.csv` — IVH por colonia
* `plantas_potabilizadoras.csv` — capa adicional para mapas

### 13.4 Limitaciones residuales

* **Precipitación mensual** ausente (descargar CLICOM/SMN si se requiere estacionalidad)
* **Antigüedad de la red** sigue siendo proxy (SACMEX no publica el dato real)

---
*Notebook generado en 2026-05-03.*
